# Classify movement in fiction
Train on labels proposed by three models, then compare predictions with a reader’s judgments on held-out works.

## Open the data
Run the setup cell once.

In [ ]:
#@title データを開く / Load the course dataset
import json
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

url = "https://nlp.lang.osaka-u.ac.jp/dh/b/materials/class7-consensus-v1.zip"
archive, _ = urlretrieve(url)
with ZipFile(archive) as bundle:
    passages = [json.loads(line) for line in bundle.open("passages.jsonl")]
    ballots = [json.loads(line) for line in bundle.open("llm-labels.jsonl")]
    reader = json.load(bundle.open("reader-labels.json"))
    with np.load(bundle.open("embeddings.npz")) as data:
        vectors = dict(zip(data["ids"], data["vectors"].astype("float32")))
    rubric = bundle.read("rubric-ja.txt").decode()
items = {p["id"]: p for p in passages}
train = [b for b in ballots if items[b["id"]]["split"] == "train" and b["label"]]
validation = [b for b in ballots if items[b["id"]]["split"] == "dev" and b["label"]]
valid = [b for b in validation if items[b["id"]].get("sampling") != "negation-cue"]
targeted = [b for b in validation if items[b["id"]].get("sampling") == "negation-cue"]
train_x = [vectors[b["id"]] for b in train]
train_y = [b["label"] for b in train]
valid_x = [vectors[b["id"]] for b in valid]
valid_y = [b["label"] for b in valid]
print("Training:", len(train), "Original validation:", len(valid), "Negation validation:", len(targeted), "Reader panel:", len(reader))

## 1. Read a disputed training passage
Read the rubric and passage. Choose your label before revealing the votes.

In [ ]:
print(rubric)
case = next(b for b in train if b["agreement"] == 2)
p = items[case["id"]]
print(p["previous"])
print(p["left"] + "【" + p["target"] + "】" + p["right"])
print(p["next"])

In [ ]:
for name, vote in case["votes"].items():
    if vote:
        print(name, vote["label"], vote["evidence"])
print("Consensus:", case["label"])

## 2. Choose a setting on validation passages
Use the saved embeddings of each marked verb in context. Run the standard weights, then change `None` to `"balanced"` and rerun. Which labels change in recall? Record your setting and reason before opening the reader panel.

In [ ]:
weights = None
model = LogisticRegression(class_weight=weights, max_iter=2000)
model.fit(train_x, train_y)
predictions = model.predict(valid_x)
present = [label for label in ["CT+", "CT-", "Uu", "NM"] if label in valid_y]
print(classification_report(valid_y, predictions, labels=present, zero_division=np.nan))

### Check CT− on the added validation passages
The original validation sample contains no CT−. Among the separate passages retrieved using negation cues, how many consensus CT− readings did your model recover?

In [ ]:
negatives = [b for b in targeted if b["label"] == "CT-"]
negative_predictions = model.predict([vectors[b["id"]] for b in negatives])
print("CT- recovered:", sum(negative_predictions == "CT-"), "/", len(negatives))

## 3. Compare with the reader
All 20 panel works were kept out of training and validation. One of the 12 random passages has a faulty target span and was deferred. Eight more were selected for rare historical Qwen labels. Read the counts separately.

In [ ]:
for cohort in ["random", "rare-9b"]:
    ids = [i for i in reader if reader[i]["cohort"] == cohort and reader[i]["label"] != "defer"]
    gold = [reader[i]["label"] for i in ids]
    predicted = model.predict([vectors[i] for i in ids])
    print(cohort, sum(predicted == gold), "/", len(ids))

Where your classifier and the reader disagree, read the target and its context. Which words support each reading?

In [ ]:
for i, reading in reader.items():
    predicted = model.predict([vectors[i]])[0]
    if reading["label"] != "defer" and reading["label"] != predicted:
        p = items[i]
        print(p["title"], p["previous"], p["left"] + "【" + p["target"] + "】" + p["right"], p["next"], sep="\n")
        print("Reader:", reading["label"], reading["evidence"], "/ Prediction:", predicted)
        break